# Results NVV-Pipeline Parameter Screening Experiment - Full-GT and Part-GT
- yml: environment.yml
- env: nvv_isolation_pipeline
- last updated: 05.04.2026


## Setup
- Imports
- Config

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from config.load_config import load_config
from config.path_factory import print_paths

from evaluation.analysis_loader import load_and_compare_experiments
from evaluation.analysis_tables import (
    build_setting_summary_table,
    build_best_all_screened_values_matrix,
    build_top_region_parameter_values_matrix,
    build_derivative_matrix,
    build_top_k_runs_table
)
from evaluation.analysis_plots import (
    plot_setting_overview,
    plot_four_panel_parameter_summary,
    plot_parameter_pair_heatmaps,
    plot_derivative_group_comparison,
    plot_top_k_runs_per_setting,
)

cfg_path_nvs_full_gt = project_root / "experiments" / "screening" / "config_test_2_files_screening_full_gt.yaml"
cfg_path_nvs_part_gt = project_root / "experiments" / "screening" / "config_test_2_files_screening_part_gt.yaml"

experiment_yaml_path = project_root / "experiments" / "screening" / "screening_test.yaml"

config = load_config(cfg_path_nvs_full_gt)
print_paths(cfg_path_nvs_full_gt)

## Collect Screening V1 Experiment Results 
- Experiment specs for comparison
- Combined Views (Tables)

In [ ]:
experiment_specs = [
    {
        "label": "2_files | full_gt",
        "cfg_path": cfg_path_nvs_full_gt,
        "dataset_name": "2_files",
        "mode": "full_gt",
    },
    {
        "label": "2_files | part_gt",
        "cfg_path": cfg_path_nvs_part_gt,
        "dataset_name": "2_files",
        "mode": "part_gt",
    },
]

analysis_bundle = load_and_compare_experiments(
    experiment_specs=experiment_specs,
    experiment_yaml_path=experiment_yaml_path,
    score_fraction=0.99,
    top_n_runs=10,
    param_names=[
        "vad_threshold",
        "vad_min_silence_ms",
        "max_duration",
        "dedup_overlap_ratio",
    ],
    param_pairs=[
        ("vad_threshold", "vad_min_silence_ms"),
    ],
    combo_top_n=10,
    top_k_rq2a_per_run=None,
)

views = analysis_bundle["combined_views"]

In [ ]:
views

In [ ]:
display(views["setting_overview"])


In [ ]:
display(
    views["parameter_value_details"][[
        "setting",
        "dataset_name",
        "mode",
        "parameter",
        "param_value_text",
        "n_runs",
        "mean_primary_score",
        "share_of_top_region",
    ]]
)

In [ ]:
display(views["pair_value_details"])

In [ ]:
views = analysis_bundle["combined_views"]

# --- collect all RQ1 tables from the four loaded experiments ---
# Filter to best_selected_set system rows for per-run comparison
rq1_all = pd.concat(
    [
        bundle["results"]["rq1"]
        for bundle in analysis_bundle["bundles_by_experiment"].values()
    ],
    ignore_index=True,
)
if "system" in rq1_all.columns:
    rq1_all = rq1_all[rq1_all["system"] == "best_selected_set"].copy()

# --- thesis-friendly summary tables ---
setting_summary_tables = build_setting_summary_table(
    views["setting_overview"],
    score_fraction=0.95,
)

best_tables = build_best_all_screened_values_matrix(
    rq1_all,
    dataset_names=[
        #"nvs38k_EN_10_categories",
        #"vocals_10_categories",
        "2_files",
    ],
    modes=[
        "full_gt",
        "part_gt",
    ],
)

top_region_parameter_values_matrix = build_top_region_parameter_values_matrix(
    views["parameter_region_summary"]
)

# Keep derivative matrices separated by metric / mode
derivative_matrix_f1 = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
)

derivative_matrix_recall = build_derivative_matrix(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
)

top_k_runs_table = build_top_k_runs_table(
    views["per_setting"],
    top_k=10,
)

print("Setting Summary (Full-GT):")
display(setting_summary_tables["full_gt"])

print("Setting Summary (Part-GT):")
display(setting_summary_tables["part_gt"])

print("\nFull-GT: Best All Screened Values Matrix:")
display(best_tables["full_gt"])

print("\nPart-GT: Best All Screened Values Matrix:")
display(best_tables["part_gt"])

print("\nTop Region Parameter Values Matrix:")
display(top_region_parameter_values_matrix)

print("\nDerivative Matrix (F1, full_gt):")
display(derivative_matrix_f1)

print("\nDerivative Matrix (Recall, part_gt):")
display(derivative_matrix_recall)

print("\nTop K Runs per Setting:")
for setting_name in top_k_runs_table["setting"].unique():
    print(f"\n=== {setting_name} ===")
    display(
        top_k_runs_table[top_k_runs_table["setting"] == setting_name]
        .reset_index(drop=True)
    )

In [ ]:
display(top_k_runs_table)
display(best_tables["full_gt"]) #toDo: kaputt?
display(best_tables["part_gt"]) #toDo: kaputt? Ansicht oben funktioniert

## Experiment Comparison Plots

In [ ]:
fig = plot_setting_overview(
    views["setting_overview"].rename(
        columns={"best_score": "plot_score"}
    ),
    score_col="plot_score",
    title=(
        "Best score per setting\n"
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_threshold",
    value_col="mean_primary_score",
    title=(
        "VAD threshold across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_min_silence_ms",
    value_col="mean_primary_score",
    title=(
        "VAD minimum silence across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="max_duration",
    value_col="mean_primary_score",
    title=(
        "Maximum NVV duration across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="dedup_overlap_ratio",
    value_col="mean_primary_score",
    title=(
        "Deduplication overlap ratio across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_parameter_pair_heatmaps(
    views["pair_value_details"],
    pair_key="vad_threshold__vad_min_silence_ms",
    value_col="mean_primary_score",
    panel_by="setting",
    title=(
        "VAD threshold × VAD minimum silence across settings\n"
        "Cell value = mean primary score within top region"
    ),
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
    panel_by="setting",
    title="Audio derivative group comparison (full_gt, metric: macro_mean_f1)",
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
    panel_by="setting",
    title="Audio derivative group comparison (part_gt, metric: macro_mean_recall)",
)
plt.show()

fig = plot_top_k_runs_per_setting(
    views["per_setting"],
    top_k=10,
    show_param_text=False,
    title=(
        "Top-k runs per setting\n"
        "Bar height = setting-specific primary metric "
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()


In [ ]:
display(top_k_runs_table[top_k_runs_table["setting"] == setting_name]
        .reset_index(drop=True)
    )

fig = plot_setting_overview(
    views["setting_overview"][
        views["setting_overview"]["mode"] == "full_gt"
    ].rename(columns={"best_score": "plot_score"}),
    score_col="plot_score",
    title="Best F1 per setting (full_gt)",
)
plt.show()

fig = plot_setting_overview(
    views["setting_overview"][
        views["setting_overview"]["mode"] == "part_gt"
    ].rename(columns={"best_score": "plot_score"}),
    score_col="plot_score",
    title="Best Recall per setting (part_gt)",
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_threshold",
    value_col="mean_primary_score",
    title=(
        "VAD threshold across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="vad_min_silence_ms",
    value_col="mean_primary_score",
    title=(
        "VAD minimum silence across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="max_duration",
    value_col="mean_primary_score",
    title=(
        "Maximum NVV duration across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_four_panel_parameter_summary(
    views["parameter_value_details"],
    parameter="dedup_overlap_ratio",
    value_col="mean_primary_score",
    title=(
        "Deduplication overlap ratio across settings\n"
        "Bar height = mean primary score within top region"
    ),
)
plt.show()

fig = plot_parameter_pair_heatmaps(
    views["pair_value_details"],
    pair_key="vad_threshold__vad_min_silence_ms",
    value_col="mean_primary_score",
    panel_by="setting",
    title=(
        "VAD threshold × VAD minimum silence across settings\n"
        "Cell value = mean primary score within top region"
    ),
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "full_gt"
    ],
    value_col="macro_mean_f1",
    panel_by="setting",
    title="Audio derivative group comparison (full_gt, metric: macro_mean_f1)",
)
plt.show()

fig = plot_derivative_group_comparison(
    views["derivative_comparison"][
        views["derivative_comparison"]["mode"] == "part_gt"
    ],
    value_col="macro_mean_recall",
    panel_by="setting",
    title="Audio derivative group comparison (part_gt, metric: macro_mean_recall)",
)
plt.show()

fig = plot_top_k_runs_per_setting(
    views["per_setting"],
    top_k=10,
    show_param_text=False,
    title=(
        "Top-k runs per setting\n"
        "Bar height = setting-specific primary metric "
        "(full_gt: macro_mean_f1, part_gt: macro_mean_recall)"
    ),
)
plt.show()


# Final Parameter Selection for Full Dataset Run

| vad_threshold | vad_min_silence_ms | max_duration | dedup_overlap_ratio |
|---|---|---|---|
| 0.20 | 50 | 6.5 | 0.5 |

**vad_threshold = 0.20**  
For the NVS dataset (`nvs38k_EN_10_categories`), all top-ranked runs use `vad_threshold = 0.20`. In both evaluation modes, the first six runs (ranks 1–6) achieve the best scores with this value: **macro_mean_f1 = 0.60** (`full_gt`) and **macro_mean_recall = 0.60** (`part_gt`). Lower-ranked runs (ranks 7–10) still use `vad_threshold = 0.20` but differ in other parameters and drop to **0.50**. For the VOCALS dataset, `part_gt` also ranks `vad_threshold = 0.20` highest (ranks 1–6 with **macro_mean_recall = 0.204167**). Only `vocals_10_categories | full_gt` prefers `0.15`, where the top runs (ranks 1–2) achieve **macro_mean_f1 = 0.218254**. Because `0.20` is consistently optimal on the NVS dataset and in `vocals_10_categories | part_gt`, it is selected as the most stable value across datasets and evaluation modes.

**vad_min_silence_ms = 50**  
On `nvs38k_EN_10_categories`, the best six runs (ranks 1–6) use `vad_min_silence_ms = 50`, achieving **macro_mean_f1 = 0.60** (`full_gt`) and **macro_mean_recall = 0.60** (`part_gt`). When the value changes to `75`, the score drops in both modes: runs 7–10 reach only **0.50**. On `vocals_10_categories | full_gt`, the top six runs (ranks 1–6) use `75` with **macro_mean_f1 between 0.218254 and 0.200000**, while runs using `50` (ranks 7–10) drop further to **0.190476**. However, in `vocals_10_categories | part_gt`, the top six runs again use `50` with **macro_mean_recall = 0.204167**, while runs using `75` appear only in ranks 7–10. Because `50` is clearly optimal on NVS and also preferred in VOCALS under `part_gt`, it is selected as the more robust value.

**max_duration = 6.5**  
Both `None` and `6.5` appear among the highest-ranked configurations. For example, in `nvs38k_EN_10_categories`, the top three runs (ranks 1–3) use `max_duration = None`, while ranks 4–6 use `6.5`, yet all six runs achieve the same score (**macro_mean_f1 = 0.60** in `full_gt`, **macro_mean_recall = 0.60` in `part_gt`). A similar pattern occurs in the VOCALS dataset, where top configurations use both `None` and `6.5` with nearly identical scores. Since the screening shows no clear performance difference, `6.5` seconds is retained as a practical safeguard to prevent overly long candidate segments.

**dedup_overlap_ratio = 0.5**  
The deduplication parameter shows minimal sensitivity in the screening results. In `nvs38k_EN_10_categories`, the first six runs (ranks 1–6) include all tested values (`0.5`, `0.7`, `0.9`) while achieving identical scores (**macro_mean_f1 = 0.60**, **macro_mean_recall = 0.60**). However, the single best run in both NVS settings (rank 1) uses `0.5`. In `vocals_10_categories | part_gt`, the top configuration (rank 1) also uses `0.5` with **macro_mean_recall = 0.204167`. Because three of the four settings select `0.5` in their best-ranked configuration and no clear performance differences are observed between the tested values, `0.5` is chosen as the final value.
